# SubSight Phase 1 gate: chrono split + cross-chunk test

Runs the honest-split comparison for pipe segmentation. Free Colab T4, 256px, seed 42. Runtime > GPU > T4 before running.

Expected output: one chrono-trained checkpoint plus IoU/Dice rows for chrono val, chrono test, and Chunk1-4 full-chunk. Paste the printed table back to fill the README gate.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/manocw/SubSight.git 2>&1 | tail -2
%cd SubSight
!pip install -r requirements.txt 2>&1 | tail -2
import torch
print('cuda:', torch.cuda.is_available())

In [ ]:
# Download SubPipe archives from Zenodo (CC-BY-4.0, verify on page). Mini first, Mini2 only if chunks are missing.
import os, zipfile
from pathlib import Path

# Exact file links from the Zenodo API record (newer version id, /content pattern).
URLS = {
    'SubPipeMini.zip': 'https://zenodo.org/api/records/12666132/files/SubPipeMini.zip/content',
    'SubPipeMini2.zip': 'https://zenodo.org/api/records/12666132/files/SubPipeMini2.zip/content',
}
Path('data').mkdir(exist_ok=True)

def fetch(name):
    out = Path('data') / name
    if not out.exists():
        print(f'downloading {name}...')
        os.system(f'wget -q --show-progress -O {out} {URLS[name]}')
    else:
        print(f'{name} already present')
    if out.stat().st_size < 10_000_000:
        raise RuntimeError(f'{name} too small ({out.stat().st_size} bytes), download failed, delete data/{name} and rerun')
    return out

def have_chunk(c):
    return Path(f'data/Chunk{c}/Segmentation').is_dir()

z = fetch('SubPipeMini.zip')
with zipfile.ZipFile(z) as f:
    f.extractall('data')

print('chunks present:', [c for c in range(5) if have_chunk(c)])
if not all(have_chunk(c) for c in range(5)):
    z2 = fetch('SubPipeMini2.zip')
    with zipfile.ZipFile(z2) as f:
        f.extractall('data')
    print('after Mini2:', [c for c in range(5) if have_chunk(c)])

In [ ]:
# Sanity: pair counts per chunk + mask check on Chunk0 (uses repo code, read-only).
import sys
sys.path.insert(0, '.')
from src.dataset import find_pairs

for c in range(5):
    root = f'data/Chunk{c}/Segmentation'
    try:
        pairs = find_pairs(root)
        print(f'Chunk{c}: {len(pairs)} pairs')
    except FileNotFoundError:
        print(f'Chunk{c}: MISSING')

!python notebooks/inspect_masks.py --root data/Chunk0/Segmentation --n 3

In [ ]:
# Chrono retrain on Chunk0: timestamp order 60/20/20, test split locked, seed 42.
!sed -i 's/split: "random"/split: "chrono"/' configs/config.yaml
!grep -A2 'train_split' configs/config.yaml
!python -m src.train --config configs/config.yaml

In [ ]:
# Gate table: chrono val first, then every available chunk full (no retrain, no leakage).
import glob, subprocess

subprocess.run(['python', '-m', 'src.evaluate', '--checkpoint', 'checkpoints/best.pth', '--num-images', '6'], check=True)
for root in sorted(glob.glob('data/Chunk*/Segmentation')):
    print(f'\n=== {root} ===')
    subprocess.run(['python', '-m', 'src.evaluate', '--checkpoint', 'checkpoints/best.pth',
                    '--data-root', root, '--full-chunk', '--num-images', '2'], check=True)

In [ ]:
# Copy these numbers into the README Phase 1 gate table. Worst-frame note included.
!ls -la checkpoints/best.pth outputs/eval_examples.png
from google.colab import files
files.download('outputs/eval_examples.png')